# 02 — Label Definition

Frost has a known physical definition, so the label is *computed*, not
learned or supplied by the dataset. This notebook builds and compares the
project's two validated label variants (see `src/label_rules.py` and the
label-definition note) on the Tier-1 (Basel) data, following Omazić et al.
(2024)'s "Method 7" as the validated formula:

1. **Broad**: `Tmin < 3.0C` alone — catches black-frost-style risk (no
   visible ice needed), more false positives against *visually observed*
   frost specifically.
2. **Strict**: `Tmin < 3.0C AND dew point < 0.0C` (dew point from Tmin and
   daily-mean RH) — the paper's best-validated rule against real station
   records, but tuned to *visible* frost by construction.

Both are computed and reported — not just one, silently.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

from data_loader import load_tier1_raw, get_city_frame, list_cities
from features import add_dew_point_features
from label_rules import build_labels_for_horizon

## Load Tier-1 data and pick a city

Using Basel for now (matches the EDA notebook and the first baseline run
already logged). `get_city_frame` also derives `humidity_pct` (0-100)
from the raw `humidity` column, which is a 0-1 FRACTION in this dataset —
easy to miss, and it silently produces a nonsense dew point if you forget
to convert it.

In [2]:
raw = load_tier1_raw('../data/raw/weather_prediction_dataset.csv')
print(f"{len(list_cities(raw))} cities available: {list_cities(raw)}")

CITY = 'BASEL'
city_df = get_city_frame(raw, CITY)
city_df[['DATE', 'temp_min', 'humidity', 'humidity_pct']].head()

18 cities available: ['BASEL', 'BUDAPEST', 'DE_BILT', 'DRESDEN', 'DUSSELDORF', 'HEATHROW', 'KASSEL', 'LJUBLJANA', 'MAASTRICHT', 'MALMO', 'MONTELIMAR', 'MUENCHEN', 'OSLO', 'PERPIGNAN', 'ROMA', 'SONNBLICK', 'STOCKHOLM', 'TOURS']


,DATE,temp_min,humidity,humidity_pct
0,2000-01-01,1.6,0.89,89.0
1,2000-01-02,2.7,0.87,87.0
2,2000-01-03,0.1,0.81,81.0
3,2000-01-04,0.5,0.79,79.0
4,2000-01-05,3.8,0.90,90.0


## Add dew point / dew-point-depression features

These live in `features.py` (the cross-repo contract module) since they're
also predictive features for the real models later, not just label
inputs.

In [3]:
city_df = add_dew_point_features(city_df, temp_col='temp_min', rh_percent_col='humidity_pct')
city_df[['DATE', 'temp_min', 'humidity_pct', 'dew_point_c', 'dew_point_depression_c']].head()

,DATE,temp_min,humidity_pct,dew_point_c,dew_point_depression_c
0,2000-01-01,1.6,89.0,-0.014640,1.614640
1,2000-01-02,2.7,87.0,0.755299,1.944701
2,2000-01-03,0.1,81.0,-2.767747,2.867747
3,2000-01-04,0.5,79.0,-2.714125,3.214125
4,2000-01-05,3.8,90.0,2.312328,1.487672


## Build (X, y) via windowing — both label variants

Horizon = 1 day ("does tomorrow dip to frost/cold-injury risk, given
today's conditions?"). `build_labels_for_horizon` slides this window
across the whole series and returns both label variants plus the
continuous target (`target_min_temp`, used later for the
regression-then-threshold models).

In [4]:
HORIZON_DAYS = 1
labeled = build_labels_for_horizon(
    city_df, horizon=HORIZON_DAYS, temp_col='temp_min', rh_pct_col='humidity_pct'
)

n = len(labeled)
n_broad = labeled['frost_label_broad'].sum()
n_strict = labeled['frost_label_strict'].sum()

print(f"Total rows: {n}")
print(f"Broad label   (Tmin<3C):            {n_broad} events  ({100*n_broad/n:.1f}%)")
print(f"Strict label  (Tmin<3C AND Td<0C):  {n_strict} events  ({100*n_strict/n:.1f}%)")

Total rows: 3653
Broad label   (Tmin<3C):            1083 events  (29.6%)
Strict label  (Tmin<3C AND Td<0C):  1011 events  (27.7%)


## Where do the two variants disagree?

By construction, `strict` can only fire where `broad` also fires (it adds
an extra AND condition) — so the interesting rows are `broad=1, strict=0`:
cold enough to injure crops, but humid enough that the computed dew point
never dropped below 0C. These are exactly the project's "black frost
candidate" nights: real cold-injury risk with no visible-ice signature,
which is why the label rule should not be gated on humidity as a
requirement.

In [5]:
assert (labeled['frost_label_strict'] <= labeled['frost_label_broad']).all(), \
    "strict should never fire where broad doesn't — check the AND logic if this fails"

divergent = labeled[(labeled['frost_label_broad'] == 1) & (labeled['frost_label_strict'] == 0)]
agreement_both = labeled[(labeled['frost_label_broad'] == 1) & (labeled['frost_label_strict'] == 1)]

print(f"Broad-only ('black frost candidate') nights: {len(divergent)}")
print(f"Nights both variants agree are frost risk:   {len(agreement_both)}")

print("\nAverage next-day conditions on broad-only nights vs both-agree nights:")
compare_cols = ['target_min_temp', 'dew_point_c', 'dew_point_depression_c', 'cloud_cover']
pd.DataFrame({
    'broad_only (candidate black frost)': divergent[compare_cols].mean(),
    'both_agree (visible-frost-like)': agreement_both[compare_cols].mean(),
}).round(2)

Broad-only ('black frost candidate') nights: 72
Nights both variants agree are frost risk:   1011

Average next-day conditions on broad-only nights vs both-agree nights:


,broad_only (candidate black frost),both_agree (visible-frost-like)
target_min_temp,2.49,-1.39
dew_point_c,-0.08,-4.39
dew_point_depression_c,2.66,3.55
cloud_cover,6.39,5.44


**Reading this table:** the broad-only nights should show a *larger*
average dew-point depression (drier air relative to temperature) than the
nights where both variants agree — that's exactly the black-frost
mechanism: cold enough to injure the crop, but too dry for the dew point
to reach the temperature and deposit visible ice. If this pattern doesn't
show up on Basel's data, that's still worth reporting honestly — it would
mean this particular station/period doesn't have much genuine black-frost
signal to illustrate the split, not that the label logic is wrong (the
logic is verified separately in `tests/test_label_rules.py`).

In [6]:
output_path = '../data/interim/basel_labeled.csv'
labeled.to_csv(output_path, index=False)
print(f"Saved {len(labeled)} labeled rows to {output_path}")
print("Columns:", list(labeled.columns))

Saved 3653 labeled rows to ../data/interim/basel_labeled.csv
Columns: ['city', 'DATE', 'cloud_cover', 'humidity', 'pressure', 'global_radiation', 'precipitation', 'sunshine', 'temp_mean', 'temp_min', 'temp_max', 'humidity_pct', 'dew_point_c', 'dew_point_depression_c', 'target_min_temp', 'frost_label_broad', 'frost_label_strict']


## Which variant to use going forward?

Per the project brief: report both, don't silently pick one. For Phase 1
baselines and Phase 3 modeling (`03_baselines.ipynb` onward), the
**broad** variant (`frost_label_broad`) is used as the primary
classification target, since under-catching black frost is the failure
mode this project is explicitly trying to avoid for an alerting system —
missing a real cold-injury event silently is worse than one extra false
alarm. The **strict** variant is kept and reported alongside it as the
higher-precision, visible-frost-validated comparison point (and is
available in the saved CSV for anyone who wants to model it instead).